In [ ]:
!pip install onnx onnxscript

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

# Sprawdzenie dostępności GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Używane urządzenie: {device}")

# KROK 1: WYBÓR 20 NAJPOPULARNIEJSZYCH KLAS
popular_classes = [
    'apple_pie', 'beef_tartare', 'caesar_salad', 'cheesecake', 'chicken_wings',
    'chocolate_cake', 'club_sandwich', 'dumplings', 'french_fries', 'garlic_bread',
    'hamburger', 'ice_cream', 'lasagna', 'macaroni_and_cheese', 'omelette',
    'pancakes', 'pizza', 'spaghetti_bolognese', 'sushi', 'waffles'
]

# Tworzymy mapowanie: stare indeksy Food101 -> nowe indeksy (0-19)
class_to_idx_mapping = {cls: idx for idx, cls in enumerate(popular_classes)}
print(f"Wybrane klasy ({len(popular_classes)}): {popular_classes}")

# KROK 2: PRZYGOTOWANIE I FILTROWANIE DANYCH
print("\n--- Krok 1: Filtrowanie zbioru danych Food101 ---")

data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# Ładowanie pełnego zbioru (z dysku Colaba)
full_train_dataset = datasets.Food101(root='.', split='train', download=True, transform=data_transforms['train'])
full_val_dataset = datasets.Food101(root='.', split='test', download=True, transform=data_transforms['val'])

def filter_indices(dataset, allowed_classes, class_to_idx_map):
    indices = []
    new_labels = {}
    for idx, target_id in enumerate(dataset._labels):
        class_name = dataset.classes[target_id]
        if class_name in allowed_classes:
            indices.append(idx)
            new_labels[idx] = class_to_idx_map[class_name]
    return indices, new_labels

train_indices, train_new_labels = filter_indices(full_train_dataset, popular_classes, class_to_idx_mapping)
val_indices, val_new_labels = filter_indices(full_val_dataset, popular_classes, class_to_idx_mapping)

train_subset = Subset(full_train_dataset, train_indices)
val_subset = Subset(full_val_dataset, val_indices)

class ModifiedSubset(torch.utils.data.Dataset):
    def __init__(self, subset, new_labels_dict):
        self.subset = subset
        self.new_labels_dict = new_labels_dict
    def __getitem__(self, index):
        img, _ = self.subset[index]
        actual_dataset_index = self.subset.indices[index]
        new_label = self.new_labels_dict[actual_dataset_index]
        return img, new_label
    def __len__(self):
        return len(self.subset)

train_dataset = ModifiedSubset(train_subset, train_new_labels)
val_dataset = ModifiedSubset(val_subset, val_new_labels)

dataloaders = {
    'train': DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2),
    'val': DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
}

num_classes = len(popular_classes)
print(f"Filtrowanie zakończone. Trening: {len(train_dataset)} zdjęć, Walidacja: {len(val_dataset)} zdjęć.")

# KROK 3: BUDOWA MODELU (TRANSFER LEARNING - FAZA 1)
print("\n--- Krok 2: Inicjalizacja modelu MobileNetV2 (Feature Extraction) ---")
try:
    model = models.mobilenet_v2(weights=models.MobileNetV2_Weights.DEFAULT)
except AttributeError:
    model = models.mobilenet_v2(pretrained=True)

# Zamrażamy całą bazę konwolucyjną
for param in model.parameters():
    param.requires_grad = False

# Podmieniamy końcówkę klasyfikatora pod nasze 20 klas
num_ftrs = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_ftrs, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier[1].parameters(), lr=0.005)

# KROK 4: TRENING FAZY 1: SZYBKIE TRENOWANIE SAMEGO KLASYFIKATORA
print("\n--- Krok 3: Trening Fazy 1 (Tylko nowa warstwa wyjściowa) ---")
num_epochs_faza1 = 2

for epoch in range(num_epochs_faza1):
    print(f'Epoka {epoch + 1}/{num_epochs_faza1} [Faza 1]')
    print('-' * 10)

    for phase in ['train', 'val']:
        if phase == 'train':
            model.train()
        else:
            model.eval()

        running_loss = 0.0
        running_corrects = 0

        for inputs, labels in dataloaders[phase]:
            inputs = inputs.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()

            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)

                if phase == 'train':
                    loss.backward()
                    optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        dataset_size = len(train_dataset) if phase == 'train' else len(val_dataset)
        epoch_loss = running_loss / dataset_size
        epoch_acc = running_corrects.double() / dataset_size
        print(f'{phase.upper()} Strata: {epoch_loss:.4f} Dokładność (Accuracy): {epoch_acc:.4f}')

# KROK 5: KONFIGURACJA FAZY 2 (FINE-TUNING)
print("\n--- Krok 4: Konfiguracja Fazy 2 (Fine-tuning) ---")
print("Odmrażanie ostatniego bloku konwolucyjnego (features[18])...")

# Odmrażamy ostatni blok ekstrakcji cech
for param in model.features[18].parameters():
    param.requires_grad = True

# Definiujemy osobne współczynniki uczenia (bardzo niskie)
optimizer_finetune = optim.Adam([
    {'params': model.features[18].parameters(), 'lr': 1e-5}, # Mikro krok dla cech
    {'params': model.classifier[1].parameters(), 'lr': 1e-4}  # Mały krok dla klasyfikatora
])

# KROK 6: TRENING FAZY 2: DOSTRAJANIE GŁĘBOKICH WARSTW
print("\n--- Krok 5: Rozpoczęcie właściwego Fine-tuningu ---")
num_epochs_faza2 = 2

for epoch in range(num_epochs_faza2):
    print(f'Epoka {epoch + 1}/{num_epochs_faza2} [Faza 2 - Fine-tuning]')
    print('-' * 10)

    for phase in ['train', 'val']:
        if phase == 'train':
            model.train()
        else:
            model.eval()

        running_loss = 0.0
        running_corrects = 0

        for inputs, labels in dataloaders[phase]:
            inputs = inputs.to(device)
            labels = labels.to(device)
            optimizer_finetune.zero_grad()

            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)

                if phase == 'train':
                    loss.backward()
                    optimizer_finetune.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        dataset_size = len(train_dataset) if phase == 'train' else len(val_dataset)
        epoch_loss = running_loss / dataset_size
        epoch_acc = running_corrects.double() / dataset_size
        print(f'{phase.upper()} Strata: {epoch_loss:.4f} Dokładność (Accuracy): {epoch_acc:.4f}')

print('Zaawansowany trening (Fine-tuning) na 20 klasach zakończony!')

# KROK 7: STABILNY EKSPORT DO FORMATU ONNX (LEGACY EXPORTER)
print("\n--- Krok 6: Eksportowanie modelu do formatu ONNX ---")
model.eval()

# Przeniesienie modelu na CPU zapobiega błędom alokacji i wersji na nowych środowiskach pytorcha
model = model.to('cpu')
dummy_input = torch.randn(1, 3, 224, 224, device='cpu')
onnx_file_path = "food_classifier_20.onnx"

torch.onnx.export(
    model,
    dummy_input,
    onnx_file_path,
    export_params=True,
    opset_version=14,  # Wyższy, stabilny zbiór operatorów
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}},
    export_modules_as_functions=False  # Blokada błędów automatycznej konwersji onnxscript
)

print(f"\nSUKCES! Plik '{onnx_file_path}' został wygenerowany poprawnie przy użyciu Fine-tuningu i jest gotowy do pobrania z plików Colaba!")